## 연습 1: 테스트 환경 설정

이 연습에서는 `pytest` 라이브러리를 설치하고 테스트할 변환 모듈을 검토합니다. 이 모듈은 원시 주문 레코드를 세 단계로 처리합니다. 유효하지 않은 항목 **필터링**, 상태 코드 **정규화**, 세금 포함 합계 **계산**입니다.

이 연습의 셀들은 모두 제공되므로 실행하기만 하면 됩니다. 여러분의 과제는 연습 2에서 시작됩니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — pytest를 설치합니다
%pip install pytest

### ✅ 제공됨: 변환 모듈 만들기

아래 셀을 실행하여 `transforms.py` 모듈을 `/tmp/lab12/`에 작성하세요. 이 모듈은 네 가지 함수를 제공합니다.

| Function | Description |
|---|---|
| `filter_valid_orders(df)` | Removes rows where `order_id` or `customer_id` is null |
| `normalize_status(df)` | Converts `status` to lowercase and strips whitespace |
| `calculate_total_with_tax(df, tax_rate=0.21)` | Adds a `total_with_tax` column |
| `run_order_pipeline(df, tax_rate=0.21)` | Runs all three transforms in sequence |

코드를 주의 깊게 검토하세요. 다음 연습에서 각 함수에 대한 테스트를 작성하게 됩니다.

In [ ]:
%%writefile /tmp/lab12/transforms.py
import pandas as pd


def filter_valid_orders(df):
    """Remove orders where order_id or customer_id is null."""
    return df.dropna(subset=['order_id', 'customer_id'])


def normalize_status(df):
    """Normalize status values to lowercase and strip whitespace."""
    df = df.copy()
    df['status'] = df['status'].str.lower().str.strip()
    return df


def calculate_total_with_tax(df, tax_rate=0.21):
    """Add a total_with_tax column based on the given tax rate."""
    df = df.copy()
    df['total_with_tax'] = (df['total'] * (1 + tax_rate)).round(2)
    return df


def run_order_pipeline(df, tax_rate=0.21):
    """Run the full order cleaning pipeline."""
    df = filter_valid_orders(df)
    df = normalize_status(df)
    df = calculate_total_with_tax(df, tax_rate)
    return df


### ✅ 제공됨: 모듈이 올바르게 로드되는지 확인

아래 셀을 실행하여 모듈을 가져오고, 테스트를 작성하기 전에 함수에 접근할 수 있는지 확인하세요.

In [ ]:
# ✅ 이 셀을 실행하세요 — 모듈이 올바르게 가져와지는지 확인합니다
import sys, os
os.makedirs('/tmp/lab12', exist_ok=True)
sys.path.insert(0, '/tmp/lab12')

from transforms import filter_valid_orders, normalize_status, calculate_total_with_tax, run_order_pipeline

print("✅ transforms.py를 성공적으로 로드했습니다.")
print("   사용 가능한 함수:", [f.__name__ for f in [filter_valid_orders, normalize_status, calculate_total_with_tax, run_order_pipeline]])

## 연습 2: 단위 테스트 작성

이 연습에서는 pytest를 사용하여 세 가지 변환 함수 각각에 대한 **단위 테스트**를 작성합니다.

아래 샘플 데이터셋을 pytest 픽스처로 사용하세요.

| order_id | customer_id | status | total |
|---|---|---|---|
| 1001 | C001 | Completed | 150.00 |
| 1002 | C002 | PENDING | 200.50 |
| `null` | C003 | cancelled | 75.00 |
| 1004 | `null` | completed | 300.00 |
| 1005 | C005 | `  Pending  ` | 125.75 |

필터링 후에는 **3개의 행**이 남습니다(`order_id` 1001, 1002, 1005 행).

> 🤖 **Genie Code:** 다음과 같이 요청하세요. *"filter_valid_orders, normalize_status, calculate_total_with_tax라는 pandas 함수에 대한 pytest 단위 테스트를 작성하도록 도와주세요. 샘플 DataFrame을 반환하는 픽스처를 포함하세요."*

아래 셀의 모든 `# TODO` 블록을 채운 뒤 실행하여 테스트 파일을 작성하세요.

In [ ]:
%%writefile /tmp/lab12/test_transforms.py
import pytest
import pandas as pd
import sys
sys.path.insert(0, '/tmp/lab12')
from transforms import filter_valid_orders, normalize_status, calculate_total_with_tax


@pytest.fixture
def sample_orders():
    return pd.DataFrame({
        'order_id':    [1001,        1002,      None,        1004,        1005        ],
        'customer_id': ['C001',      'C002',    'C003',      None,        'C005'      ],
        'status':      ['Completed', 'PENDING', 'cancelled', 'completed', '  Pending  '],
        'total':       [150.00,      200.50,    75.00,       300.00,      125.75      ],
    })


# ── filter_valid_orders에 대한 테스트 ─────────────────────────────────────────────

def test_filter_removes_null_order_id(sample_orders):
    result = filter_valid_orders(sample_orders)
    assert result['order_id'].isnull().sum() == 0


def test_filter_removes_null_customer_id(sample_orders):
    result = filter_valid_orders(sample_orders)
    assert result['customer_id'].isnull().sum() == 0


def test_filter_retains_correct_row_count(sample_orders):
    result = filter_valid_orders(sample_orders)
    assert len(result) == 3


# ── normalize_status에 대한 테스트 ────────────────────────────────────────────────

def test_normalize_status_lowercases_values(sample_orders):
    result = normalize_status(sample_orders)
    assert all(result['status'] == result['status'].str.lower())


def test_normalize_status_strips_whitespace(sample_orders):
    result = normalize_status(sample_orders)
    assert all(result['status'] == result['status'].str.strip())


# ── calculate_total_with_tax에 대한 테스트 ────────────────────────────────────────

def test_calculate_total_with_tax_adds_column(sample_orders):
    result = calculate_total_with_tax(sample_orders)
    assert 'total_with_tax' in result.columns


def test_calculate_total_with_tax_default_rate(sample_orders):
    result = calculate_total_with_tax(sample_orders)
    assert result.iloc[0]['total_with_tax'] == round(150.00 * 1.21, 2)


def test_calculate_total_with_tax_custom_rate(sample_orders):
    result = calculate_total_with_tax(sample_orders, tax_rate=0.10)
    assert result.iloc[1]['total_with_tax'] == round(200.50 * 1.10, 2)


### ✅ 단위 테스트 실행

아래 셀을 실행하여 pytest로 테스트를 수행하세요. **8개의 테스트가 모두 통과해야 합니다.** 실패하는 테스트가 있으면 오류 출력을 주의 깊게 읽고 실패한 테스트 로직을 수정한 뒤 `%%writefile` 셀을 다시 실행하고 아래 셀을 다시 실행하세요.

In [ ]:
# ✅ 이 셀을 실행하세요 — 모든 단위 테스트를 실행합니다
import sys
sys.path.insert(0, '/tmp/lab12')
import pytest

retcode = pytest.main(['/tmp/lab12/test_transforms.py', '-v', '-p', 'no:cacheprovider'])
assert retcode == 0, "❌ 단위 테스트가 실패했습니다. 위의 출력을 검토하고 실패한 테스트를 수정하세요."